
## Denormalization:

- **Denormalization** is the process of `combining data from multiple normalized tables into a single, wider table to reduce joins`. 
- improve **query performance** for `analytics and reporting`.

**Real-time example: E-commerce sales analytics**

**Suppose an online shopping company stores its data in a star schema.**


### A. Source Tables:

**1. Sales Fact Table**

| SaleID | CustomerID | ProductID | Quantity | Revenue |
|---|---|---|---|---|
| 1 | 101 | 201 | 2 | 2000 |
| 2 | 102 | 202 | 1 | 1500 |
| 3 | 101 | 202 | 3 | 4500 |

**2. Customer Dimension Table**

| CustomerID | CustomerName | City |
|---|---|---|
| 101 | Arun | Chennai |
| 102 | Bala | Madurai |

**3. Product Dimension Table**

| ProductID | ProductName | Category |
|---|---|---|
| 201 | Laptop | Electronics |
| 202 | Shoes | Fashion |




## Create the dataframes:

In [0]:
sales_fact_df = spark.createDataFrame([
    (1, 101, 201, 2, 2000.0),
    (2, 102, 202, 1, 1500.0),
    (3, 101, 202, 3, 4500.0)
], ["SaleID", "CustomerID", "ProductID", "Quantity", "Revenue"])

customer_dim_df = spark.createDataFrame([
    (101, "Arun", "Chennai"),
    (102, "Bala", "Madurai")
], ["CustomerID", "CustomerName", "City"])

product_dim_df = spark.createDataFrame([
    (201, "Laptop", "Electronics"),
    (202, "Shoes", "Fashion")
], ["ProductID", "ProductName", "Category"])

## Denormalize using joins:

In [0]:
wide_sales_df = (
    sales_fact_df
    .join(customer_dim_df, "CustomerID", "left")
    .join(product_dim_df, "ProductID", "left")
)

wide_sales_df.show()


## Perform analytics on the wide dataset:

**Suppose the business wants revenue by city and product category.**

In [0]:
from pyspark.sql import functions as F

wide_sales_df.groupBy("City", "Category").agg(
    F.sum("Revenue").alias("TotalRevenue")
).show()


## Save the denormalized dataset:

In [0]:
wide_sales_df.write.mode("overwrite").parquet(
    "/Volumes/workspace/default/data_volume"
)


## Review the Output:

In [0]:
wide_sales_df.show()